In [ ]:
from rotary_embedding_torch import RotaryEmbedding
from transformers import GPT2TokenizerFast
import torch
import torch.nn as nn
import torch.functional as F
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seq_len = 512
batch = 16
# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = len(tokenizer)
print(f"Vocabulary size: {vocab_size}")

def encode(text):
    """Encode text using GPT-2 tokenizer"""
    return tokenizer.encode(text, add_special_tokens=False)

def decode(indices):
    """Decode token indices back to text"""
    return tokenizer.decode(indices)

In [ ]:
#7500 more steps
import numpy as np
import os


chunk_size = 1_000_000  # characters
tokens = []
if not os.path.exists("/content/tokens.npy"):
  with open("/content/drive/MyDrive/Combined.txt", "r", encoding="utf-8") as f:
      while True:
          chunk = f.read(chunk_size)
          if not chunk:
              break
          tokens.extend(encode(chunk))

  tokens = np.array(tokens, dtype=np.int32)
  invalid = np.where((tokens < 0) | (tokens >= vocab_size))[0]
  print(f"Found {len(invalid)} invalid tokens at positions {invalid[:10]}...")
  np.save("tokens.npy", tokens)
  tokens = np.memmap("tokens.npy", dtype=np.int32, mode="r")
else:
   tokens = np.memmap("tokens.npy", dtype=np.int32, mode="r")
n = int(0.9*len(tokens))
train_data = tokens[:n]
val_data = tokens[n:]
def get_batch(split):
    data = train_data if split == 'train' else val_data
    max_start = len(data) - seq_len - 1
    ix = torch.randint(0,max_start , (batch,))

    x = torch.stack([
        torch.from_numpy(data[i:i+seq_len].astype(np.int64))
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(data[i+1:i+seq_len+1].astype(np.int64))
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)
    return x, y




In [ ]:

d_model = 512
heads = 8
n_layers = 8


class Postitional_Embeddings(nn.Module):
  def __init__(self,seq_len,d_model):
      super().__init__()
      self.pos = nn.Embedding(seq_len,d_model)
  def forward(self,x):
     seq_len = x.size(1)
     positions = torch.arange(0,seq_len, device=x.device)
     return self.pos(positions).unsqueeze(0)

class TransformerBlock(nn.Module):
  def __init__(self,d_model, heads,max_seq_len):
    super().__init__()
    self.norm1 = nn.RMSNorm(d_model)
    self.att = nn.MultiheadAttention(num_heads=heads,embed_dim=d_model,batch_first=True,dropout=0.2)
    self.norm2 = nn.RMSNorm(d_model)
    self.mlp = nn.Sequential(
        nn.Linear(d_model,d_model*4),
        nn.GELU(),
        nn.Linear(d_model*4,d_model),
    )
    mask = torch.triu(torch.full((max_seq_len, max_seq_len), float('-inf')), diagonal=1)

    self.register_buffer("causal_mask", mask)

  def forward(self,x):
   B, T, _ = x.shape
   h = self.norm1(x)
   attn_mask = self.causal_mask[:T, :T]
   x = x + self.att(h,h,h,need_weights=False, attn_mask=attn_mask)[0]
   x = x + self.mlp(self.norm2(x))
   return x
class Transformer(nn.Module):
    def __init__(self, d_model, heads, n_layers,seq_len):
        super().__init__()

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, heads,max_seq_len=seq_len)
            for _ in range(n_layers)
        ])

        self.norm = nn.RMSNorm(d_model)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.norm(x)
class GPT(nn.Module):
  def __init__(self,vocab_size, seq_len, d_model, heads, n_layers):
     super().__init__()
     self.token_emb = nn.Embedding(vocab_size,d_model)
     self.pos_emb = Postitional_Embeddings(seq_len, d_model)
     self.transformer = Transformer(d_model, heads, n_layers,seq_len)
     self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
     self.lm_head.weight = self.token_emb.weight

  def forward(self, idx):

        # Token + position embedding
        x = self.token_emb(idx)                 # (batch, seq_len, d_model)
        x = x + self.pos_emb(idx)               # (batch, seq_len, d_model)
        x = self.transformer(x)                 # (batch, seq_len, d_model)

        # Project to vocab
        logits = self.lm_head(x)                # (batch, seq_len, vocab_size)

        return logits
model = GPT(vocab_size, seq_len, d_model, heads, n_layers)


def init_weights(module):
    if isinstance(module, nn.Linear):
        # Standard Xavier initialization for linear layers
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        # Small normal initialization for embeddings
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.LayerNorm) or isinstance(module, nn.RMSNorm):
        # LayerNorm weights to 1, bias to 0
        nn.init.ones_(module.weight)
model.apply(init_weights)

m = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1
)


In [ ]:
criterion = nn.CrossEntropyLoss()

def train_step():
    m.train()

    x, y = get_batch('train')           # x,y: (B, T)

    logits = m(x)                       # (B, T, vocab)
    B, T, V = logits.shape

    loss = criterion(
        logits.view(B*T, V),
        y.view(B*T)
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    optimizer.step()

    return loss.item()
@torch.no_grad()
def eval_step():
    m.eval()

    x, y = get_batch('val')
    logits = m(x)

    B, T, V = logits.shape
    loss = criterion(
        logits.view(B*T, V),
        y.view(B*T)
    )
    return loss.item()
max_iters = 1000
eval_interval = 100

for step in range(max_iters):
    loss = train_step()

    if step % 10 == 0:
        print(f"step {step} | train loss {loss:.4f}")

    if step % eval_interval == 0:
        val_loss = eval_step()
        print(f"--- val loss {val_loss:.4f} ---")


In [ ]:
import torch.nn.functional as F
import time

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.5, top_k=None):
    model.eval()
    idx = idx.to(next(model.parameters()).device)  # move to same device as model
    for _ in range(max_new_tokens):
        # crop idx to last `seq_len` tokens to fit model input
        idx_cond = idx[:, -seq_len:]
        # forward pass
        logits = m(idx_cond)              # (1, seq_len, vocab_size)
        logits = logits[:, -1, :]             # (1, vocab) -> last token

        # apply temperature
        logits = logits / temperature

        # optionally restrict to top_k
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        # convert to probabilities
        probs = F.softmax(logits, dim=-1)

        # sample next token
        next_idx = torch.multinomial(probs, num_samples=1)
        # append to sequence
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

# starting token(s)
start_text = "He stood there"
start_tokens = torch.tensor([encode(start_text)], dtype=torch.long)

# generate 50 new tokens
generated_tokens = generate(m, start_tokens, max_new_tokens=100, temperature=0.7, top_k=50)

generated_text = decode(generated_tokens[0].tolist())

words = generated_text.split(" ")
for w in words:
    print(w + " ", end="", flush=True)
    time.sleep(0.02 if w != "\n" else 0.2)